In [1]:
import pandas as pd

In [2]:
train=pd.read_csv("train.csv")
test=pd.read_csv("test.csv")
sample=pd.read_csv("sample_submission.csv")

In [3]:
print("训练集大小：",train.shape)
print("测试集大小:",test.shape)

display(train.head())
train.info()

print("标签比例:")
print(train["Will_Buy_EV"].value_counts(normalize=True))

print("各列缺失数量:")
print(train.isna().sum())

训练集大小： (668665, 15)
测试集大小: (286571, 14)


,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 668665 entries, 0 to 668664
Data columns (total 15 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           668665 non-null  int64  
 1   Age                          668665 non-null  int64  
 2   Annual_Income_USD            668665 non-null  float64
 3   Daily_Commute_km             668665 non-null  float64
 4   Number_of_Cars_Owned         668665 non-null  int64  
 5   Charging_Stations_Near_Home  668665 non-null  int64  
 6   Charging_Stations_Near_Work  668665 non-null  int64  
 7   Environmental_Concern_Level  668665 non-null  float64
 8   Gender                       668665 non-null  object 
 9   City_Type                    668665 non-null  object 
 10  Current_Car_Type             668665 non-null  object 
 11  Home_Charging_Possible       668665 non-null  object 
 12  Subsidy_Available            668665 non-null  object 
 13 

<h3 style="color: #3498db;">  Step 1: 确定输入X和标签y  </h3>

In [ ]:
X=train.drop(columns=["id","Will_Buy_EV",])
y=train["Will_Buy_EV"].map({
    "No":0,
    "Yes":1
})


print(X.shape)
print(y.value_counts(normalize=True))

(668665, 14)
Will_Buy_EV
0    0.825355
1    0.174645
Name: proportion, dtype: float64


<h3 style="color: #3498db;">  Step 2: 找类别特征  </h3>

In [20]:
cat_cols=X.select_dtypes(
    include=["object"]
).columns.tolist()

print(cat_cols)

['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


<h3 style="color: #3498db;">  Step 3: 划分验证集  </h3>

In [21]:
from sklearn.model_selection import train_test_split

X_train,X_vaild,y_train,y_vaild=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

model=CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    eval_metric="AUC",
    cat_features=cat_cols,
    verbose=100
)

model.fit(
    X_train,
    y_train,
    eval_set=(X_vaild,y_vaild),
    early_stopping_rounds=100
)

0:	test: 0.9318313	best: 0.9318313 (0)	total: 237ms	remaining: 3m 56s
100:	test: 0.9397957	best: 0.9397957 (100)	total: 26.5s	remaining: 3m 56s
200:	test: 0.9402734	best: 0.9402734 (200)	total: 53.8s	remaining: 3m 33s
300:	test: 0.9406514	best: 0.9406514 (300)	total: 1m 21s	remaining: 3m 8s
400:	test: 0.9409555	best: 0.9409555 (400)	total: 1m 47s	remaining: 2m 40s
500:	test: 0.9410873	best: 0.9410875 (499)	total: 2m 17s	remaining: 2m 17s
600:	test: 0.9411882	best: 0.9411882 (600)	total: 2m 46s	remaining: 1m 50s
700:	test: 0.9412327	best: 0.9412330 (699)	total: 3m 16s	remaining: 1m 23s
800:	test: 0.9412830	best: 0.9412830 (800)	total: 3m 52s	remaining: 57.7s
900:	test: 0.9413113	best: 0.9413138 (895)	total: 4m 27s	remaining: 29.4s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9413138033
bestIteration = 895

Shrink model to first 896 iterations.


CatBoostClassifier(cat_features=['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level'], depth=8, eval_metric='AUC', iterations=1000, learning_rate=0.05, verbose=100)

: 

In [29]:
print(model.best_iteration_)

942


In [30]:
pred = model.predict_proba(X_vaild)[:,1]

print(
    roc_auc_score(y_vaild,pred)
)

0.8894920390415262


In [31]:
X_test=test.drop(columns=["id"])
test_pred=model.predict_proba(X_test)[:,1]

CatBoostError: Bad value for num_feature[non_default_doc_idx=0,feature_idx=12]="Low": Cannot convert 'Low' to float

In [ ]:
sample.head()

,id,Will_Buy_EV
0,668665,0.174645
1,668666,0.174645
2,668667,0.174645
3,668668,0.174645
4,668669,0.174645


In [ ]:
submission=sample.copy()
submission["Will_Buy_EV"]=test_pred
submission.head()

,id,Will_Buy_EV
0,668665,0.010764
1,668666,0.020420
2,668667,0.003990
3,668668,0.001837
4,668669,0.022663
